# Notebook 03 — Shortest Path Algorithms

**Goal:** Verify custom Dijkstra and A* against NetworkX reference, benchmark runtime on the Kigali graph, and visualize sample paths on a Folium map.

**Requires:** `data/kigali_enriched.graphml`

**Produces:** `results/shortest_path_benchmark.csv`, `results/sample_routes.html`

**Phase:** 5

## Cell 1 — Imports and load enriched graph

In [9]:
import sys, os, time, random
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import folium

from src.graph import load_enriched_graph, get_node_coordinates
from src.algorithms import dijkstra, astar

os.makedirs('../results', exist_ok=True)

G = load_enriched_graph()
nodes = list(G.nodes())
print(f'Graph: {len(nodes):,} nodes, {len(G.edges()):,} edges')
print('Algorithms imported: dijkstra, astar')

Loading enriched graph from: data/kigali_enriched.graphml
Loaded — 19,022 nodes, 50,411 edges
Graph: 19,022 nodes, 50,411 edges
Algorithms imported: dijkstra, astar


## Cell 2 — Sample 20 random source-destination pairs

In [10]:
random.seed(42)

# Sample from the largest strongly connected component to guarantee paths exist
scc = max(nx.strongly_connected_components(G), key=len)
scc_nodes = list(scc)
print(f'Largest SCC: {len(scc_nodes):,} nodes ({len(scc_nodes)/len(nodes)*100:.1f}% of graph)')

pairs = []
while len(pairs) < 20:
    s = random.choice(scc_nodes)
    t = random.choice(scc_nodes)
    if s != t:
        pairs.append((s, t))

print(f'Sampled {len(pairs)} source-destination pairs from SCC')

Largest SCC: 19,015 nodes (100.0% of graph)
Sampled 20 source-destination pairs from SCC


## Cell 3 — Correctness: custom Dijkstra vs NetworkX reference

In [11]:
print('Verifying custom Dijkstra against NetworkX reference...')
print('{:<6}  {:>14}  {:>14}  {:>10}'.format('Pair', 'Custom (min)', 'NetworkX (min)', 'Match'))
print('-' * 52)

mismatches = 0
for i, (src, tgt) in enumerate(pairs):
    path_custom, cost_custom = dijkstra(G, src, tgt, weight='composite_weight')
    cost_nx = nx.dijkstra_path_length(G, src, tgt, weight='composite_weight')
    match = abs(cost_custom - cost_nx) < 1e-6
    if not match:
        mismatches += 1
    print('{:<6}  {:>14.6f}  {:>14.6f}  {:>10}'.format(i+1, cost_custom, cost_nx, str(match)))

print(f'\nMismatches: {mismatches}/20')
assert mismatches == 0, f'{mismatches} mismatches between custom Dijkstra and NetworkX!'
print('✓ Custom Dijkstra matches NetworkX on all 20 pairs')

Verifying custom Dijkstra against NetworkX reference...
Pair      Custom (min)  NetworkX (min)       Match
----------------------------------------------------
1            16.784164       16.784164        True
2            18.950300       18.950300        True
3            32.212954       32.212954        True
4            24.158904       24.158904        True
5            28.405976       28.405976        True
6            30.262618       30.262618        True
7            22.039048       22.039048        True
8            12.130833       12.130833        True
9             8.499594        8.499594        True
10           45.004578       45.004578        True
11            6.213053        6.213053        True
12           17.346322       17.346322        True
13           14.943423       14.943423        True
14           17.309792       17.309792        True
15           12.534204       12.534204        True
16           39.592192       39.592192        True
17           33.225190  

## Cell 4 — Correctness: A* vs Dijkstra

In [12]:
print('Verifying A* against custom Dijkstra...')
print('{:<6}  {:>14}  {:>14}  {:>10}'.format('Pair', 'Dijkstra (min)', 'A* (min)', 'Match'))
print('-' * 52)

mismatches = 0
for i, (src, tgt) in enumerate(pairs):
    _, cost_dijk = dijkstra(G, src, tgt, weight='composite_weight')
    _, cost_astar = astar(G, src, tgt, weight='composite_weight')
    match = abs(cost_dijk - cost_astar) < 1e-6
    if not match:
        mismatches += 1
    print('{:<6}  {:>14.6f}  {:>14.6f}  {:>10}'.format(i+1, cost_dijk, cost_astar, str(match)))

print(f'\nMismatches: {mismatches}/20')
assert mismatches == 0, f'{mismatches} mismatches between A* and Dijkstra!'
print('✓ A* matches Dijkstra on all 20 pairs')

Verifying A* against custom Dijkstra...
Pair    Dijkstra (min)        A* (min)       Match
----------------------------------------------------
1            16.784164       16.784164        True
2            18.950300       18.950300        True
3            32.212954       32.212954        True
4            24.158904       24.158904        True
5            28.405976       28.405976        True
6            30.262618       30.262618        True
7            22.039048       22.039048        True
8            12.130833       12.130833        True
9             8.499594        8.499594        True
10           45.004578       45.004578        True
11            6.213053        6.213053        True
12           17.346322       17.346322        True
13           14.943423       14.943423        True
14           17.309792       17.309792        True
15           12.534204       12.534204        True
16           39.592192       39.592192        True
17           33.225190       33.225190  

## Cell 5 — Runtime benchmark

In [13]:
def benchmark_algo(algo_fn, pairs, label, n_reps=3):
    times = []
    for src, tgt in pairs:
        reps = []
        for _ in range(n_reps):
            t0 = time.perf_counter()
            algo_fn(G, src, tgt, weight='composite_weight')
            reps.append((time.perf_counter() - t0) * 1000)
        times.append(min(reps))   # best of n_reps
    print(f'{label:<20}  mean={np.mean(times):6.2f}ms  median={np.median(times):6.2f}ms  p95={np.percentile(times,95):6.2f}ms')
    return times

def nx_dijkstra(G, src, tgt, weight):
    return nx.dijkstra_path_length(G, src, tgt, weight=weight)

print('Runtime benchmark on 20 pairs (best of 3 reps each):\n')
print('{:<20}  {:>12}  {:>12}  {:>10}'.format('Algorithm', 'Mean (ms)', 'Median (ms)', 'P95 (ms)'))
print('-' * 60)

times_dijk  = benchmark_algo(dijkstra,    pairs, 'Custom Dijkstra')
times_astar = benchmark_algo(astar,       pairs, 'Custom A*')
times_nx    = benchmark_algo(nx_dijkstra, pairs, 'NetworkX Dijkstra')

print(f'\nA* speedup over custom Dijkstra: {np.mean(times_dijk)/np.mean(times_astar):.2f}x')

Runtime benchmark on 20 pairs (best of 3 reps each):

Algorithm                Mean (ms)   Median (ms)    P95 (ms)
------------------------------------------------------------
Custom Dijkstra       mean= 31.61ms  median= 37.72ms  p95= 48.25ms
Custom A*             mean= 14.76ms  median= 13.37ms  p95= 31.88ms
NetworkX Dijkstra     mean= 19.86ms  median= 23.90ms  p95= 30.03ms

A* speedup over custom Dijkstra: 2.14x


## Cell 6 — Visualize 3 sample paths on a Folium map

In [14]:
COLORS = ['#e63946', '#2a9d8f', '#f4a261']
sample_pairs = pairs[:3]

# Center map on Kigali
m = folium.Map(location=[-1.9441, 30.0619], zoom_start=13, tiles='OpenStreetMap')

for idx, (src, tgt) in enumerate(sample_pairs):
    path, cost = dijkstra(G, src, tgt, weight='composite_weight')
    if path is None:
        continue

    color = COLORS[idx]
    coords = [get_node_coordinates(G, n) for n in path]   # (lat, lon) tuples

    # Draw route line
    folium.PolyLine(
        locations=coords,
        color=color,
        weight=4,
        opacity=0.85,
        tooltip=f'Route {idx+1}: {cost:.3f} min'
    ).add_to(m)

    # Start marker
    folium.CircleMarker(
        location=coords[0],
        radius=8, color=color, fill=True, fill_opacity=1.0,
        tooltip=f'Route {idx+1} start'
    ).add_to(m)

    # End marker
    folium.CircleMarker(
        location=coords[-1],
        radius=8, color=color, fill=True, fill_opacity=0.3,
        tooltip=f'Route {idx+1} end ({cost:.3f} min)'
    ).add_to(m)

    print(f'Route {idx+1}: {len(path)} nodes, {cost:.3f} min')

output_path = '../results/sample_routes.html'
m.save(output_path)
print(f'\nMap saved to {output_path}')
m   # render inline in JupyterLab

Route 1: 87 nodes, 16.784 min
Route 2: 97 nodes, 18.950 min
Route 3: 150 nodes, 32.213 min

Map saved to ../results/sample_routes.html


## Cell 7 — Write benchmark CSV

In [15]:
records = []
for algo_label, times in [
    ('custom_dijkstra',  times_dijk),
    ('custom_astar',     times_astar),
    ('networkx_dijkstra', times_nx),
]:
    records.append({
        'algorithm':  algo_label,
        'n_pairs':    len(times),
        'mean_ms':    round(np.mean(times),   4),
        'median_ms':  round(np.median(times), 4),
        'p95_ms':     round(np.percentile(times, 95), 4),
        'min_ms':     round(np.min(times),    4),
        'max_ms':     round(np.max(times),    4),
    })

df = pd.DataFrame(records)
csv_path = '../results/shortest_path_benchmark.csv'
df.to_csv(csv_path, index=False)
print(df.to_string(index=False))
print(f'\nSaved to {csv_path}')

        algorithm  n_pairs  mean_ms  median_ms  p95_ms  min_ms  max_ms
  custom_dijkstra       20  31.6126    37.7187 48.2525  2.9850 51.8757
     custom_astar       20  14.7641    13.3679 31.8796  1.0198 36.2919
networkx_dijkstra       20  19.8595    23.8994 30.0293  1.7055 31.4071

Saved to ../results/shortest_path_benchmark.csv


## Cell 8 — Phase 5 validation

In [16]:
assert os.path.exists('../results/shortest_path_benchmark.csv'), 'CSV missing'
assert os.path.exists('../results/sample_routes.html'),          'HTML map missing'

df_check = pd.read_csv('../results/shortest_path_benchmark.csv')
assert len(df_check) == 3,           'Expected 3 rows in benchmark CSV'
assert 'mean_ms' in df_check.columns, 'mean_ms column missing'
assert (df_check['n_pairs'] == 20).all(), 'Expected 20 pairs per algorithm'

print('✓ results/shortest_path_benchmark.csv exists and is valid')
print('✓ results/sample_routes.html exists')
print('\n✓ Phase 5 validation passed. Proceed to notebook 04.')

✓ results/shortest_path_benchmark.csv exists and is valid
✓ results/sample_routes.html exists

✓ Phase 5 validation passed. Proceed to notebook 04.
